# Sat2Cap Inference Demo

This notebook demonstrates how to run inference with the pretrained **Sat2Cap** model for text-image similarity matching on satellite imagery.

Sat2Cap learns to associate natural language text descriptions with satellite images, enabling zero-shot text-based mapping from overhead imagery.

**Pretrained models** are available on HuggingFace: [MVRL Remote Sensing Foundation Models](https://huggingface.co/collections/MVRL/remote-sensing-foundation-models)

**Paper:** [Sat2Cap: Mapping Fine-Grained Textual Descriptions from Satellite Images (CVPRW EarthVision 2024)](https://openaccess.thecvf.com/content/CVPR2024W/EarthVision/html/Dhakal_Sat2Cap_Mapping_Fine-Grained_Textual_Descriptions_from_Satellite_Images_CVPRW_2024_paper.html)

## 1. Install Dependencies

Install the required packages if you haven't already.

In [ ]:
# Install required packages
# !pip install torch torchvision transformers huggingface_hub Pillow matplotlib

## 2. Load the Pretrained Sat2Cap Model

Download and load the pretrained Sat2Cap model from HuggingFace Hub.

In [ ]:
import torch
from huggingface_hub import hf_hub_download

# Add the repo root to the Python path so we can import sat2cap modules
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from sat2cap.utils.load_model import load_sat2cap

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Download and load the pretrained Sat2Cap model from HuggingFace Hub
# The model is hosted at https://huggingface.co/MVRL/sat2cap
model = load_sat2cap(repo_id='MVRL/sat2cap', filename='sat2cap.ckpt')
model = model.to(device)
model.eval()
print('Sat2Cap model loaded successfully!')

## 3. Prepare the Image Preprocessor

Sat2Cap uses CLIP's image preprocessing pipeline with satellite-image-specific normalization.

In [ ]:
import torchvision.transforms as transforms
from PIL import Image

def _convert_image_to_rgb(image):
    return image.convert('RGB')

# Preprocessing pipeline for overhead/satellite images
# Uses satellite-specific normalization statistics
satellite_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(size=(224, 224)),
    _convert_image_to_rgb,
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.3670, 0.3827, 0.3338),
        std=(0.2209, 0.1975, 0.1988)
    )
])

def preprocess_image(image_path):
    """Load and preprocess a satellite image for Sat2Cap inference."""
    img = Image.open(image_path)
    return satellite_transform(img).unsqueeze(0)  # add batch dimension

print('Image preprocessor ready.')

## 4. Load a Sample Satellite Image

We use a sample satellite image for this demo. You can replace this with any satellite/overhead image you have.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

# Option 1: Use an image from HuggingFace Hub
# The Sat2Cap model card includes sample satellite images
sample_image_path = hf_hub_download(
    repo_id='MVRL/sat2cap',
    filename='sample_satellite_image.jpg'
)

# Option 2: Use a local image — uncomment and update path:
# sample_image_path = 'path/to/your/satellite_image.jpg'

# Display the image
img = Image.open(sample_image_path)
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis('off')
plt.title('Sample Satellite Image')
plt.tight_layout()
plt.show()
print(f'Image size: {img.size}')

## 5. Extract Satellite Image Embeddings

Run the satellite image through the Sat2Cap overhead image encoder to get embeddings.

In [ ]:
# Preprocess and encode the satellite image
img_tensor = preprocess_image(sample_image_path).to(device)

with torch.no_grad():
    # imo_encoder is the overhead/satellite image encoder
    normalized_img_embedding, unnormalized_img_embedding = model.imo_encoder(img_tensor)

print(f'Satellite image embedding shape: {normalized_img_embedding.shape}')
print(f'Embedding norm: {normalized_img_embedding.norm().item():.4f}  (should be ~1.0 for normalized)')

## 6. Text-Image Similarity Matching

Compute cosine similarity between text prompts and the satellite image embedding.
This demonstrates Sat2Cap's ability to do **zero-shot text-based querying** of satellite images.

In [ ]:
from transformers import AutoTokenizer, CLIPTextModelWithProjection

# Load CLIP text encoder (same backbone used by Sat2Cap)
clip_model_name = 'openai/clip-vit-base-patch32'
tokenizer = AutoTokenizer.from_pretrained(clip_model_name)
text_model = CLIPTextModelWithProjection.from_pretrained(clip_model_name).to(device).eval()

# Define text prompts to compare against the satellite image
text_prompts = [
    'a photo of a residential neighborhood',
    'a photo of a forest',
    'a photo of farmland and agricultural fields',
    'a photo of a beach and coastline',
    'a photo of a city center with tall buildings',
    'a photo of a river or lake',
    'a photo of a highway or road',
    'a photo of a desert',
    'a photo of an airport',
    'a photo of a sports stadium',
]

# Tokenize and encode the text prompts
with torch.no_grad():
    tokenized = tokenizer(text_prompts, padding=True, return_tensors='pt').to(device)
    text_outputs = text_model(**tokenized)
    text_embeddings = text_outputs.text_embeds
    # Normalize text embeddings
    text_embeddings = text_embeddings / text_embeddings.norm(p=2, dim=-1, keepdim=True)

print(f'Text embeddings shape: {text_embeddings.shape}')

In [ ]:
# Compute cosine similarity between the satellite image and each text prompt
with torch.no_grad():
    similarities = (normalized_img_embedding @ text_embeddings.T).squeeze(0)
    similarities = similarities.cpu().numpy()

# Display results
print('Text-Image Similarity Scores:')
print('-' * 55)
sorted_indices = similarities.argsort()[::-1]
for idx in sorted_indices:
    print(f'{similarities[idx]:.4f}  |  {text_prompts[idx]}')

## 7. Visualize Similarity Scores

In [ ]:
import numpy as np

# Sort by similarity score
sorted_indices = similarities.argsort()[::-1]
sorted_prompts = [text_prompts[i] for i in sorted_indices]
sorted_scores = similarities[sorted_indices]

# Shorten prompt labels for the plot
short_labels = [p.replace('a photo of ', '') for p in sorted_prompts]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: satellite image
axes[0].imshow(img)
axes[0].axis('off')
axes[0].set_title('Satellite Image', fontsize=13)

# Right: similarity bar chart
colors = ['#2196F3' if s == sorted_scores.max() else '#90CAF9' for s in sorted_scores]
bars = axes[1].barh(range(len(short_labels)), sorted_scores, color=colors)
axes[1].set_yticks(range(len(short_labels)))
axes[1].set_yticklabels(short_labels, fontsize=10)
axes[1].invert_yaxis()  # highest score on top
axes[1].set_xlabel('Cosine Similarity', fontsize=11)
axes[1].set_title('Text-Image Similarity (Sat2Cap)', fontsize=13)
axes[1].axvline(x=0, color='gray', linewidth=0.8, linestyle='--')

# Add score labels on bars
for i, (bar, score) in enumerate(zip(bars, sorted_scores)):
    axes[1].text(
        bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
        f'{score:.3f}', va='center', fontsize=9
    )

plt.tight_layout()
plt.savefig('sat2cap_similarity_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nTop match: "{sorted_prompts[0]}" (score: {sorted_scores[0]:.4f})')

## 8. Batch Inference on Multiple Images

Sat2Cap can also process multiple images at once. Here's how to run batch inference.

In [ ]:
def get_image_embeddings(image_paths, model, transform, device, batch_size=16):
    """Compute Sat2Cap embeddings for a list of satellite image paths.
    
    Args:
        image_paths (list[str]): Paths to satellite images.
        model: Loaded Sat2Cap model.
        transform: Image preprocessing transform.
        device: Torch device.
        batch_size (int): Number of images per batch.
    
    Returns:
        torch.Tensor: Normalized image embeddings of shape (N, 512).
    """
    all_embeddings = []
    model.eval()
    
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i + batch_size]
        batch_tensors = torch.stack([
            transform(Image.open(p)) for p in batch_paths
        ]).to(device)
        
        with torch.no_grad():
            normalized_embeddings, _ = model.imo_encoder(batch_tensors)
        
        all_embeddings.append(normalized_embeddings.cpu())
    
    return torch.cat(all_embeddings, dim=0)


def get_text_embeddings(text_prompts, tokenizer, text_model, device):
    """Compute CLIP text embeddings for a list of text prompts.
    
    Args:
        text_prompts (list[str]): Text descriptions to embed.
        tokenizer: CLIP tokenizer.
        text_model: CLIP text model.
        device: Torch device.
    
    Returns:
        torch.Tensor: Normalized text embeddings of shape (N, 512).
    """
    with torch.no_grad():
        tokenized = tokenizer(text_prompts, padding=True, return_tensors='pt').to(device)
        text_outputs = text_model(**tokenized)
        text_embeddings = text_outputs.text_embeds
        text_embeddings = text_embeddings / text_embeddings.norm(p=2, dim=-1, keepdim=True)
    return text_embeddings.cpu()


# Example: rank text prompts for a single image
image_paths = [sample_image_path]  # replace with your list of image paths
img_embeddings = get_image_embeddings(image_paths, model, satellite_transform, device)
txt_embeddings = get_text_embeddings(text_prompts, tokenizer, text_model, device)

# Similarity matrix: (num_images x num_prompts)
similarity_matrix = img_embeddings @ txt_embeddings.T
print(f'Similarity matrix shape: {similarity_matrix.shape}')
print(f'Best matching prompt for image 0: "{text_prompts[similarity_matrix[0].argmax()]}"')